# Chapter 7: Rotations, Rotors, and Versors

This standalone replacement notebook covers the operational center of Chapter 7: reflections become sandwich products, pairs of reflections become rotations, and products of vectors become reusable transformation operators. The source anchor for this pass is the textbook chapter whose printed pages run from 167 through 212. In the PDF used for this workspace, that material begins on PDF page 197 with the Chapter 7 heading and ends on PDF page 242 with the final user-interface rotation example; PDF page 243 starts Chapter 8. The notebook prose here is original study material, not a transcription.

The goal is not to imitate the book's route line by line. Instead, the notebook turns the chapter into a set of executable habits. We will keep one small geometric-algebra implementation in view and repeatedly check the same claim from different angles: an orthogonal transformation is most naturally stored as the geometric element that performs it. A vector can describe a mirror, a bivector can describe a plane of rotation, a rotor can compose with another rotor, and the same sandwiching pattern can act on vectors, blades, and mixed multivectors.

By the end you should be able to read a rotor as both a number-like element and an operator. You should also be able to explain why the sign of a rotor matters even when it leaves the visible rotated vector unchanged, why unit complex numbers and unit quaternions sit naturally inside rotor algebra, and why versors are more than a clever notation for matrices. The saved artifacts are intended to be inspected outside the notebook as reusable figures, so each major computational section writes either Plotly HTML, image data, or a JSON check file under `artifacts/chapter-07`.


## Setup

The setup cell imports the compact algebra tools used throughout the course and the Chapter 7 helper functions added for this replacement. The helper module deliberately avoids magic: a rotor is still a multivector, `apply_rotor(X, R)` still computes `R X reverse(R)`, and the reflection helpers are just the corresponding one-vector or blade sandwich products. Keeping those formulas close to the notebook is useful because Chapter 7 is about changing the status of products. Earlier chapters used products to build subspaces and measure projections; here the product itself becomes a first-class transformation.

A small warning about conventions is worth stating at the beginning. A Euclidean rotor in a plane `B` is written here as `cos(angle/2) - B sin(angle/2)` when `B` is a unit bivector. With the basis orientation used by `utils.ga`, this sends `e1` to `e2` for a positive quarter turn in the `e1^e2` plane. Other books and libraries may put the sign in the exponential the other way. That is not a contradiction; it is a convention about oriented planes and active versus passive rotations. The sanity checks below pin down the convention used in this notebook.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "utils" / "ga").exists():
        PROJECT_ROOT = candidate
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.artifacts import display_artifact, save_image, save_json, save_plotly_html
from utils.chapter07_rotors import (
    E2,
    E3,
    E4,
    apply_even_versor,
    apply_odd_versor,
    apply_rotor,
    coords_to_vector,
    normalize_versor,
    quaternion_product,
    quaternion_wxyz_to_rotor,
    real_vector_julia,
    reflect_in_hyperplane,
    reflect_in_line,
    reflect_vector_in_subspace,
    rotor_exp,
    rotor_from_plane_angle,
    rotor_log,
    rotor_power,
    rotor_to_matrix,
    rotor_to_quaternion_wxyz,
    unit_vector,
    vector_to_coords,
)

ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
TOPIC = "chapter-07"
artifact_paths: list[Path] = []


def remember(path: Path) -> Path:
    artifact_paths.append(path)
    return path


def rel(path: Path) -> str:
    return path.relative_to(PROJECT_ROOT).as_posix()


def mv_close(left, right, tol: float = 1e-8) -> bool:
    return left.almost_equal(right, tol=tol)


def arrow_trace(name, start, end, color, width=7):
    start = np.asarray(start, dtype=float)
    end = np.asarray(end, dtype=float)
    return go.Scatter3d(
        x=[start[0], end[0]],
        y=[start[1], end[1]],
        z=[start[2], end[2]],
        mode="lines+markers",
        line={"color": color, "width": width},
        marker={"size": [2, 5], "color": color},
        name=name,
    )


def vector_trace(name, vector, color, width=7):
    return arrow_trace(name, [0, 0, 0], vector_to_coords(vector.grade(1)), color, width)


np.set_printoptions(precision=6, suppress=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Artifact root: {ARTIFACT_ROOT / TOPIC}")


## Reflections Make Subspaces Act

A reflection is the first place where Chapter 7 changes the meaning of a geometric element. A vector can still be a directed segment, but it can also be the normal of a mirror hyperplane or the direction of an axis line. The difference is not stored in a new data type; it is stored in how the vector is used. For a line through the origin, the sandwich `a X a^{-1}` keeps the component of a vector parallel to `a` and flips the perpendicular component. For a hyperplane with normal `a`, the grade-involuted version flips the normal component and keeps the hyperplane component. In three dimensions that second operation is the ordinary mirror reflection in a plane.

The point is not that every reflection must be implemented by hand. The point is that a reflection is no longer a table of coordinate formulas. Once the reflecting element is invertible, the same product pattern can be applied to a vector, a blade, or a mixed multivector. That is what makes the construction scale. If a triangle edge, a surface normal, and an oriented area blade all pass through the same transformation code, there are fewer places for sign and orientation bugs to hide.

The figure below compares a line reflection and a hyperplane reflection for the same vector. The line-reflected vector stays on the other side of the chosen line, while the hyperplane-reflected vector behaves like a mirror image across the `e1-e2` plane. We save this as a Plotly artifact because reflection pictures are most useful when they can be orbited interactively.


In [ ]:
e1, e2, e3 = E3.basis()
x = coords_to_vector([1.35, 0.65, 0.85], E3)
line_direction = unit_vector([1.0, 1.0, 0.0], E3)
plane_normal = e3

line_reflected = reflect_in_line(x, line_direction).grade(1)
plane_reflected = reflect_in_hyperplane(x, plane_normal).grade(1)
plane_blade = e1.wedge(e2)
plane_reflected_from_blade = reflect_vector_in_subspace(x, plane_blade)

assert mv_close(plane_reflected, plane_reflected_from_blade)
assert abs(x.norm2() - line_reflected.norm2()) < 1e-9
assert abs(x.norm2() - plane_reflected.norm2()) < 1e-9

plane_grid = go.Surface(
    x=[[-1.6, 1.6], [-1.6, 1.6]],
    y=[[-1.6, -1.6], [1.6, 1.6]],
    z=[[0, 0], [0, 0]],
    opacity=0.22,
    colorscale=[[0, "#B7E4C7"], [1, "#B7E4C7"]],
    showscale=False,
    name="mirror plane",
)
line_coords = vector_to_coords(line_direction)
line_trace = go.Scatter3d(
    x=[-1.7 * line_coords[0], 1.7 * line_coords[0]],
    y=[-1.7 * line_coords[1], 1.7 * line_coords[1]],
    z=[0, 0],
    mode="lines",
    line={"color": "#364FC7", "width": 5},
    name="line reflector",
)
fig = go.Figure(
    data=[
        plane_grid,
        line_trace,
        vector_trace("x", x, "#212529"),
        vector_trace("line reflection", line_reflected, "#F08C00"),
        vector_trace("plane reflection", plane_reflected, "#2B8A3E"),
    ]
)
fig.update_layout(
    title="A vector used as a line reflector or as a plane normal",
    scene={"aspectmode": "cube", "xaxis_title": "e1", "yaxis_title": "e2", "zaxis_title": "e3"},
    margin={"l": 0, "r": 0, "t": 40, "b": 0},
)
reflection_path = remember(
    save_plotly_html(fig, TOPIC, "reflections", "line-plane-reflections.html", root=ARTIFACT_ROOT)
)
display_artifact(reflection_path, height=520)
print(rel(reflection_path))


## Two Reflections Become One Rotor

The chapter's central construction is almost disarmingly simple: perform one reflection, then another. Geometrically, the result is a rotation. Algebraically, the two reflecting vectors multiply into one even element. If the two reflecting directions are unit vectors `a` and `b`, the composite operator can be stored as `R = b a`. Applying the two reflections separately gives the same result as applying the one sandwich product `R X reverse(R)`.

This is the first reason rotors feel different from matrices. A matrix can certainly rotate a vector, but it does not remember the factorization into geometric mirrors. A rotor is an element of the algebra, so it composes by the same product that built it. That means the expression for a new operator has the same kind of object on both sides of the equals sign. There is no jump from vectors to arrays and back again.

The half-angle is the main conceptual trap. If `a` and `b` are separated by `theta/2` in their plane, the double reflection rotates vectors by `theta`. The product `b a` contains a scalar part and a bivector part, so it records both the relative angle and the oriented plane. In the code below we reflect a vector in one line, reflect the result in a second line, and compare that with the rotor product. The saved artifact draws all three vectors so the equality is visible before the assertion confirms it.


In [ ]:
angle = np.deg2rad(58.0)
a = unit_vector([1.0, 0.0, 0.0], E3)
b = unit_vector([np.cos(angle / 2.0), np.sin(angle / 2.0), 0.0], E3)
probe = coords_to_vector([1.0, 0.35, 0.55], E3)

first_reflection = reflect_in_line(probe, a).grade(1)
second_reflection = reflect_in_line(first_reflection, b).grade(1)
R_double = normalize_versor(b.gp(a))
rotor_result = apply_rotor(probe, R_double).grade(1)

assert mv_close(second_reflection, rotor_result)
assert abs(R_double.gp(R_double.reverse()).scalar_value() - 1.0) < 1e-9

arc_t = np.linspace(0, angle, 60)
arc = np.column_stack([np.cos(arc_t), np.sin(arc_t), np.zeros_like(arc_t)])
fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=arc[:, 0],
        y=arc[:, 1],
        z=arc[:, 2],
        mode="lines",
        name="rotation angle",
        line={"color": "#868E96", "width": 4},
    )
)
fig.add_trace(vector_trace("probe", probe, "#212529"))
fig.add_trace(vector_trace("after first reflection", first_reflection, "#F08C00"))
fig.add_trace(vector_trace("after second reflection", second_reflection, "#2F9E44"))
fig.add_trace(vector_trace("rotor result", rotor_result, "#1971C2", width=4))
fig.add_trace(vector_trace("a", a, "#7048E8", width=5))
fig.add_trace(vector_trace("b", b, "#C2255C", width=5))
fig.update_layout(
    title="Two line reflections equal one rotor sandwich",
    scene={"aspectmode": "cube", "xaxis_title": "e1", "yaxis_title": "e2", "zaxis_title": "e3"},
    margin={"l": 0, "r": 0, "t": 40, "b": 0},
)
double_path = remember(
    save_plotly_html(fig, TOPIC, "double-reflection", "double-reflection-rotor.html", root=ARTIFACT_ROOT)
)
display_artifact(double_path, height=520)
print({"rotor": repr(R_double), "artifact": rel(double_path)})


## Rotor Sign And The 4pi Sense

The sandwich product cannot distinguish `R` from `-R` when it acts on a single vector: both signs cancel because one copy appears on the left and the reverse appears on the right. It is tempting to call that a redundancy and move on. Chapter 7 asks us to be more careful. The two signs encode different histories of the rotation process. A physical vector returns to its original direction after a `2pi` turn, but the rotor tracing that process reaches `-1`, not `+1`. Only after `4pi` does the rotor itself return to the identity.

This is the double-cover behavior of rotors. It is not a numerical nuisance. It is the reason interpolation, composition, and animation code must decide whether it is following the shortest visible rotation or a rotation with a remembered sense. In practice the sign choice is a path choice. If an application only cares about final orientations, it can flip signs to keep interpolation short. If it cares about a continuous process, such as a controller or a tracked hand orientation, the sign carries useful continuity information.

The plot below follows a vector and its rotor through a full `4pi` sweep in the `e1^e2` plane. The vector endpoint closes after `2pi`. The rotor scalar and bivector coordinates do not close until the second turn. This is the executable version of the chapter's qualitative point: a rotor is not merely the final pose; it is an oriented route through the space of rotations.


In [ ]:
angles = np.linspace(0.0, 4.0 * np.pi, 241)
rotor_scalars = []
rotor_bivectors = []
vector_xy = []
plane = e1.wedge(e2)
for phi in angles:
    R = rotor_from_plane_angle(plane, phi)
    rotor_scalars.append(R.grade(0).scalar_value())
    rotor_bivectors.append(R.terms.get(0b011, 0.0))
    vector_xy.append(vector_to_coords(apply_rotor(e1, R).grade(1))[:2])
vector_xy = np.asarray(vector_xy)

R_2pi = rotor_from_plane_angle(plane, 2.0 * np.pi)
R_4pi = rotor_from_plane_angle(plane, 4.0 * np.pi)
assert mv_close(apply_rotor(e1, R_2pi).grade(1), e1)
assert R_2pi.grade(0).scalar_value() < -0.999999
assert R_4pi.grade(0).scalar_value() > 0.999999

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Vector endpoint", "Rotor coordinates"),
    specs=[[{"type": "xy"}, {"type": "xy"}]],
)
fig.add_trace(
    go.Scatter(x=vector_xy[:, 0], y=vector_xy[:, 1], mode="lines", name="rotated e1"),
    row=1,
    col=1,
)
fig.add_trace(go.Scatter(x=[1], y=[0], mode="markers", marker={"size": 9}, name="start"), row=1, col=1)
fig.add_trace(go.Scatter(x=angles / np.pi, y=rotor_scalars, mode="lines", name="scalar"), row=1, col=2)
fig.add_trace(
    go.Scatter(x=angles / np.pi, y=rotor_bivectors, mode="lines", name="e1^e2 coefficient"),
    row=1,
    col=2,
)
fig.update_xaxes(title_text="e1", row=1, col=1)
fig.update_yaxes(title_text="e2", scaleanchor="x", scaleratio=1, row=1, col=1)
fig.update_xaxes(title_text="angle / pi", row=1, col=2)
fig.update_yaxes(title_text="coordinate", row=1, col=2)
fig.update_layout(title="Visible rotations close after 2pi; rotors close after 4pi", height=460)
fourpi_path = remember(
    save_plotly_html(fig, TOPIC, "rotor-sense", "rotor-4pi-sense.html", root=ARTIFACT_ROOT)
)
display_artifact(fourpi_path, height=500)
print({"R_2pi": repr(R_2pi), "R_4pi": repr(R_4pi), "artifact": rel(fourpi_path)})


## Composition Is Multiplication, With Order

Rotors compose by the geometric product. If `R1` is applied first and `R2` second, the combined rotor is `R2 R1`. The order is the same order used by ordinary function composition: the operator nearest the vector acts first after the expression is expanded. This order matters in three dimensions because rotation planes need not commute. A turn in the `e1^e2` plane followed by a turn in the `e2^e3` plane is generally not the same as doing those turns in the opposite order.

This is another place where the algebra pays for itself. A rotation matrix also composes by multiplication, but a rotor product remains an element that can be logged, interpolated, negated, factored, or applied to blades without changing representation. The product has a scalar part and bivector parts that can be interpreted as a new rotation plane and angle in three dimensions. When we convert to matrices below, we do it only as a diagnostic: the matrix columns are the images of the basis vectors under the rotor sandwich.

The artifact draws the transformed basis triads for the two possible orders. The JSON check file records determinants, norm preservation, and a simple noncommutativity measurement. A determinant near `+1` is a useful reminder that these even versors preserve orientation, unlike single hyperplane reflections.


In [ ]:
R12 = rotor_from_plane_angle(e1.wedge(e2), np.deg2rad(75.0))
R23 = rotor_from_plane_angle(e2.wedge(e3), np.deg2rad(50.0))
R_23_after_12 = normalize_versor(R23.gp(R12))
R_12_after_23 = normalize_versor(R12.gp(R23))

M_a = rotor_to_matrix(R_23_after_12)
M_b = rotor_to_matrix(R_12_after_23)
commutator_size = np.linalg.norm(M_a - M_b)
assert abs(np.linalg.det(M_a) - 1.0) < 1e-9
assert abs(np.linalg.det(M_b) - 1.0) < 1e-9
assert commutator_size > 0.1

colors = ["#E03131", "#2F9E44", "#1971C2"]
fig = go.Figure()
for matrix, prefix, shift in [(M_a, "R23 R12", np.array([-1.5, 0, 0])), (M_b, "R12 R23", np.array([1.5, 0, 0]))]:
    for i, color in enumerate(colors):
        fig.add_trace(arrow_trace(f"{prefix} e{i + 1}", shift, shift + matrix[:, i], color, width=6))
fig.update_layout(
    title="Rotor composition is order-sensitive in 3-D",
    scene={"aspectmode": "cube", "xaxis_title": "e1", "yaxis_title": "e2", "zaxis_title": "e3"},
    margin={"l": 0, "r": 0, "t": 40, "b": 0},
)
composition_path = remember(
    save_plotly_html(fig, TOPIC, "composition", "composition-triads.html", root=ARTIFACT_ROOT)
)
composition_json = remember(
    save_json(
        {
            "det_R23_after_R12": float(np.linalg.det(M_a)),
            "det_R12_after_R23": float(np.linalg.det(M_b)),
            "matrix_difference_norm": float(commutator_size),
            "R23_after_R12": repr(R_23_after_12),
            "R12_after_R23": repr(R_12_after_23),
        },
        TOPIC,
        "composition",
        "composition-checks.json",
        root=ARTIFACT_ROOT,
    )
)
display_artifact(composition_path, height=520)
print(rel(composition_json))


## Complex Numbers And Quaternions Inside Rotor Algebra

In a real plane, the unit pseudoscalar squares to `-1`, so the even subalgebra has the same multiplication table as complex numbers. That observation is more than a mnemonic. It says that ordinary complex multiplication is a special case of real geometric multiplication in a selected plane. The usual complex unit is not a mysterious external object; it is the oriented area element of the plane. The rotor sandwich uses half angles, while direct complex multiplication uses the full angle, but the geometric content is the same planar rotation.

In three dimensions, unit quaternions sit in the same role. A quaternion's scalar part and three imaginary coordinates correspond to a rotor's scalar and three bivector coordinates. The bivectors are dual to axes: a rotation around the `e3` axis is stored in the `e1^e2` plane, and similarly for the other axes. This notebook uses the coordinate convention `(w, x, y, z) <-> w - x e23 + y e13 - z e12`. The signs come from the rotor convention introduced in the setup cell.

The code below performs two checks. First it verifies that planar rotor action agrees with multiplying a complex number by a unit complex number. Then it multiplies two 3-D rotors and verifies that the mapped quaternion coordinates satisfy the Hamilton product. The lesson is not that we should always abandon complex numbers or quaternions. The lesson is that rotors explain where their multiplication rules come from and show how to generalize beyond their native dimensions.


In [ ]:
f1, f2 = E2.basis()
I2 = f1.wedge(f2)
z_vec = coords_to_vector([0.55, 1.15], E2)
theta = np.deg2rad(40.0)
R2 = rotor_from_plane_angle(I2, theta)
rotated_vec = apply_rotor(z_vec, R2).grade(1)
complex_expected = (0.55 + 1.15j) * np.exp(1j * theta)
assert np.linalg.norm(vector_to_coords(rotated_vec) - np.array([complex_expected.real, complex_expected.imag])) < 1e-9

Q1_rotor = rotor_from_plane_angle(e1.wedge(e2), np.deg2rad(35.0))
Q2_rotor = rotor_from_plane_angle(e2.wedge(e3), np.deg2rad(-62.0))
Q_total = normalize_versor(Q2_rotor.gp(Q1_rotor))
q1 = rotor_to_quaternion_wxyz(Q1_rotor)
q2 = rotor_to_quaternion_wxyz(Q2_rotor)
q_total_from_rotor = rotor_to_quaternion_wxyz(Q_total)
q_total_hamilton = quaternion_product(q2, q1)
assert np.linalg.norm(q_total_from_rotor - q_total_hamilton) < 1e-9
assert mv_close(quaternion_wxyz_to_rotor(q_total_hamilton), Q_total)

complex_quat_json = remember(
    save_json(
        {
            "complex_input": [0.55, 1.15],
            "complex_rotated_by_rotor": vector_to_coords(rotated_vec).tolist(),
            "complex_expected": [float(complex_expected.real), float(complex_expected.imag)],
            "q1_wxyz": q1.tolist(),
            "q2_wxyz": q2.tolist(),
            "q2_times_q1_wxyz": q_total_hamilton.tolist(),
            "rotor_product": repr(Q_total),
        },
        TOPIC,
        "complex-quaternion",
        "complex-quaternion-links.json",
        root=ARTIFACT_ROOT,
    )
)
print(rel(complex_quat_json))


## Exponentials, Logs, And Interpolation

A pure Euclidean rotor can be written as `exp(-B/2)`, where `B` is a bivector angle: its direction is the rotation plane and its magnitude is the visible rotation angle. This is the rotor analogue of writing a unit complex number as an exponential. The logarithm reverses the process. Given a unit rotor, it extracts the principal bivector angle that generates it, up to the usual branch choices. Branch choices are not bookkeeping trivia; they determine which path interpolation follows.

Interpolation with rotors is especially clean. To move from `R0` to `R1`, form the relative rotor `Delta = R1 reverse(R0)`, raise `Delta` to a fractional power by scaling its logarithm, and then multiply back onto `R0`. In symbols, the interpolated rotor is `Delta^t R0`. This is the geometric-algebra version of spherical linear interpolation. It keeps the operator normalized, moves through meaningful intermediate rotations, and can be applied to every grade of object without rebuilding separate formulas for points, normals, planes, or areas.

The next cell uses a skew rotation plane in three dimensions, verifies that `log` and `exp` invert each other on this example, and traces the interpolated path of both a vector and an oriented area blade. The blade path is represented by its dual normal for plotting, but the actual transformed object is still computed as a bivector sandwich.


In [ ]:
B_skew = e1.wedge(e2) + 0.55 * e2.wedge(e3)
R_target = rotor_from_plane_angle(B_skew, 2.15)
B_recovered = rotor_log(R_target)
R_recovered = rotor_exp(B_recovered)
assert mv_close(R_recovered, R_target)

sample_vector = coords_to_vector([1.0, 0.2, 0.35], E3)
sample_blade = (e1 + 0.2 * e2).wedge(e3)
ts = np.linspace(0.0, 1.0, 80)
vector_path = []
blade_normal_path = []
for t in ts:
    Rt = rotor_power(R_target, float(t))
    vt = apply_rotor(sample_vector, Rt).grade(1)
    Bt = apply_rotor(sample_blade, Rt).grade(2)
    vector_path.append(vector_to_coords(vt))
    blade_normal_path.append(vector_to_coords(Bt.dual().grade(1)))
vector_path = np.asarray(vector_path)
blade_normal_path = np.asarray(blade_normal_path)

fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=vector_path[:, 0],
        y=vector_path[:, 1],
        z=vector_path[:, 2],
        mode="lines",
        name="vector path",
        line={"color": "#1971C2", "width": 6},
    )
)
fig.add_trace(
    go.Scatter3d(
        x=blade_normal_path[:, 0],
        y=blade_normal_path[:, 1],
        z=blade_normal_path[:, 2],
        mode="lines",
        name="dual normal of rotated blade",
        line={"color": "#C2255C", "width": 6},
    )
)
fig.add_trace(vector_trace("initial vector", sample_vector, "#212529", width=5))
fig.add_trace(vector_trace("final vector", apply_rotor(sample_vector, R_target).grade(1), "#2F9E44", width=5))
fig.update_layout(
    title="Rotor logarithms make interpolation an operator path",
    scene={"aspectmode": "cube", "xaxis_title": "e1", "yaxis_title": "e2", "zaxis_title": "e3"},
    margin={"l": 0, "r": 0, "t": 40, "b": 0},
)
interp_path = remember(
    save_plotly_html(fig, TOPIC, "interpolation", "rotor-log-interpolation.html", root=ARTIFACT_ROOT)
)
display_artifact(interp_path, height=520)
print({"log_R": repr(B_recovered), "artifact": rel(interp_path)})


## Subspaces As Operators

Once reflections are written as sandwich products, the wall between objects and operators becomes porous. A line, plane, or higher-grade blade can be a drawable subspace in one expression and a reflection operator in the next. This is not a type error. It is the point of the representation. A direct plane blade in three dimensions, for example, can reflect a vector in that plane. If the plane is transformed by a rotor, the reflected result transforms consistently because the sandwich product preserves the algebraic structure that defines the construction.

The phrase "subspace as operator" should not be read as "every blade is secretly a matrix." A blade carries orientation, grade, and metric information that a plain matrix does not. When it acts as a reflector, those data determine which components flip and which remain. The action extends to compound objects by outermorphism: reflect the factors and wedge them, or use the corresponding sandwich structure, and the result represents the same transformed subspace.

The checks below use a tilted plane as a mirror. We reflect two vectors and their wedge, confirm that inner products are preserved, and save a compact JSON report. Then we check the structure-preserving property for an even versor: applying the rotor to a geometric product gives the same result as transforming each factor first and multiplying the transformed factors. That identity is the reason versor code can preserve constructions rather than merely move coordinates.


In [ ]:
plane_reflector = unit_vector([1.0, -0.35, 0.2], E3).wedge(unit_vector([0.25, 0.7, 1.0], E3))
u = coords_to_vector([0.8, -0.1, 0.4], E3)
w = coords_to_vector([-0.2, 0.9, 0.35], E3)
U = u.wedge(w)

u_ref = reflect_vector_in_subspace(u, plane_reflector)
w_ref = reflect_vector_in_subspace(w, plane_reflector)
U_ref_by_factors = u_ref.wedge(w_ref)

inner_before = u.scalar_product(w).scalar_value()
inner_after = u_ref.scalar_product(w_ref).scalar_value()
area_before = abs(U.gp(U.reverse()).scalar_value())
area_after = abs(U_ref_by_factors.gp(U_ref_by_factors.reverse()).scalar_value())
assert abs(inner_before - inner_after) < 1e-9
assert abs(area_before - area_after) < 1e-9

A = coords_to_vector([0.6, 0.1, -0.2], E3) + 0.4 * e1.wedge(e3)
B = coords_to_vector([-0.3, 0.8, 0.15], E3) + 0.2 * e2.wedge(e3)
left = apply_even_versor(A.gp(B), R_target)
right = apply_even_versor(A, R_target).gp(apply_even_versor(B, R_target))
assert mv_close(left, right)

odd_normal = unit_vector([0.0, 1.0, 1.0], E3)
odd_reflected_u = apply_odd_versor(u, odd_normal).grade(1)
assert abs(odd_reflected_u.norm2() - u.norm2()) < 1e-9

subspace_json = remember(
    save_json(
        {
            "plane_reflector": repr(plane_reflector),
            "inner_before": float(inner_before),
            "inner_after": float(inner_after),
            "area_before": float(area_before),
            "area_after": float(area_after),
            "even_versor_preserves_geometric_product": True,
            "odd_reflection_preserves_vector_norm": True,
        },
        TOPIC,
        "subspaces-as-operators",
        "subspace-versor-checks.json",
        root=ARTIFACT_ROOT,
    )
)
print(rel(subspace_json))


## Versors Beyond Simple 3-D Rotors

A versor is a product of invertible vectors used as an operator. Even unit versors are rotors; odd versors represent orientation-reversing orthogonal transformations such as reflections. In three dimensions it is easy to fall into the habit of thinking that every useful rotor is just a scalar plus one bivector direction. That intuition is too small for the general theory. In four dimensions, a rotation can turn two orthogonal planes at once. The resulting rotor may contain a scalar part, bivector parts, and a grade-4 part, while still being a perfectly valid even unit versor.

This matters for computer science because high-dimensional geometric data do appear: configuration spaces, projective and conformal models, color transforms, optimization states, and simulation constraints all exceed ordinary 3-D intuition. A representation that is built from the algebra rather than from a hand-coded `3 x 3` formula has somewhere to go when the dimension changes. The implementation still deserves care, especially for performance, but the mathematical interface remains stable: products compose operators and sandwiches apply them.

The next cell constructs a four-dimensional rotor as the product of commuting rotations in the `e1^e2` and `e3^e4` planes. The grade report shows that the composite rotor is not merely scalar plus one bivector. It still has unit versor norm, and it still preserves vector length when applied by the even versor product.


In [ ]:
g1, g2, g3, g4 = E4.basis()
R12_4 = rotor_from_plane_angle(g1.wedge(g2), np.deg2rad(70.0))
R34_4 = rotor_from_plane_angle(g3.wedge(g4), np.deg2rad(-35.0))
R4 = normalize_versor(R34_4.gp(R12_4))
probe4 = coords_to_vector([0.8, -0.1, 0.45, 0.9], E4)
probe4_rotated = apply_even_versor(probe4, R4).grade(1)

grade_report = {str(grade): repr(R4.grade(grade)) for grade in sorted(R4.grades())}
assert set(R4.grades()) == {0, 2, 4}
assert abs(R4.gp(R4.reverse()).scalar_value() - 1.0) < 1e-9
assert abs(probe4.norm2() - probe4_rotated.norm2()) < 1e-9

versor4_json = remember(
    save_json(
        {
            "rotor_grades": grade_report,
            "rotor": repr(R4),
            "input_norm2": float(probe4.norm2()),
            "output_norm2": float(probe4_rotated.norm2()),
            "input_vector": vector_to_coords(probe4).tolist(),
            "output_vector": vector_to_coords(probe4_rotated).tolist(),
        },
        TOPIC,
        "versors",
        "four-dimensional-versor.json",
        root=ARTIFACT_ROOT,
    )
)
print(rel(versor4_json))


## Julia Sets From Real Vector Products

The chapter ends by showing that a familiar complex iteration can be rewritten in real geometric algebra. If a complex number is represented by a real vector relative to a chosen real-axis direction `e`, the square in the Julia iteration can be expressed as a geometric product `x e x`. In two dimensions this gives the same update as the usual complex square. The benefit is conceptual: the formula no longer depends on a separate complex-number type, so it suggests how the iteration might be lifted to vector spaces where the word "imaginary" is no longer the right guide.

The implementation below computes a 2-D Julia escape image using the real-vector update. It saves both an interactive Plotly heatmap and a PNG image. The PNG is intentionally generated from the numeric escape counts rather than from a copied textbook figure. You can change the constant `c`, the iteration budget, or the viewing extent and rerun the cell to create a different artifact. The visible result is a fractal, but the underlying lesson is the same as for rotations: the geometric product supplies the operation that a specialized number system used to hide.

The 3-D version is more expensive to visualize well, so this notebook keeps the generated figure 2-D and treats the higher-dimensional extension as a design prompt. The helper function returns raw counts, which makes the artifact useful for tests and for future rendering experiments.


In [ ]:
counts = real_vector_julia(width=420, height=300, c=(-0.72, 0.19), extent=1.62, max_iter=54)
normalized = counts / counts.max()
rgb = np.zeros((*counts.shape, 3), dtype=np.uint8)
rgb[..., 0] = np.clip(28 + 190 * normalized, 0, 255).astype(np.uint8)
rgb[..., 1] = np.clip(38 + 150 * np.sqrt(normalized), 0, 255).astype(np.uint8)
rgb[..., 2] = np.clip(70 + 120 * (1.0 - normalized), 0, 255).astype(np.uint8)

julia_png = remember(save_image(rgb, TOPIC, "julia-vector-fractal", "real-vector-julia.png", root=ARTIFACT_ROOT))
fig = go.Figure(data=go.Heatmap(z=counts, colorscale="Viridis", showscale=True))
fig.update_layout(
    title="Julia escape counts from the real-vector update x e x + c",
    xaxis={"visible": False},
    yaxis={"visible": False, "scaleanchor": "x", "scaleratio": 1},
    height=520,
    margin={"l": 0, "r": 0, "t": 45, "b": 0},
)
julia_html = remember(save_plotly_html(fig, TOPIC, "julia-vector-fractal", "real-vector-julia.html", root=ARTIFACT_ROOT))
display_artifact(julia_png, width=520)
display_artifact(julia_html, height=560)
print({"png": rel(julia_png), "html": rel(julia_html), "max_count": float(counts.max())})


## Final Sanity Checks

This last cell is intentionally plain. It gathers the artifacts written by the notebook, verifies that they exist, and writes a final JSON manifest. It also repeats the core mathematical invariants in one place: rotor application preserves vector norm, double reflection agrees with the rotor sandwich, rotor composition is noncommutative for the chosen 3-D example, the quaternion correspondence matches Hamilton multiplication, the exponential/logarithm round trip works on the chosen rotor, the even versor preserves a geometric product, and the four-dimensional rotor preserves length despite having a grade-4 component.

These checks are not a proof of the whole chapter. They are guardrails for an executable study notebook. If a future edit changes a sign convention, a basis ordering, or a helper function, at least one of these assertions should complain. That is the practical benefit of turning the chapter into a seed-style notebook rather than leaving it as a static outline: the explanations and the algebra have to keep each other honest.


In [ ]:
checks = {
    "pdf_span_verified": "printed pp. 167-212; PDF pages 197-242 by chapter headings",
    "artifact_count": len(artifact_paths),
    "all_artifacts_exist": all(path.exists() and path.stat().st_size > 0 for path in artifact_paths),
    "double_reflection_equals_rotor": mv_close(second_reflection, rotor_result),
    "rotor_2pi_is_negative_identity_operator": bool(R_2pi.grade(0).scalar_value() < -0.999999),
    "rotor_4pi_returns_positive_identity": bool(R_4pi.grade(0).scalar_value() > 0.999999),
    "composition_order_matters": bool(commutator_size > 0.1),
    "quaternion_product_matches_rotor_product": bool(np.linalg.norm(q_total_from_rotor - q_total_hamilton) < 1e-9),
    "rotor_log_exp_round_trip": mv_close(R_recovered, R_target),
    "even_versor_preserves_product": mv_close(left, right),
    "four_dimensional_rotor_has_grade4_part": 4 in R4.grades(),
    "julia_grid_shape": list(counts.shape),
}
assert checks["all_artifacts_exist"]
assert checks["double_reflection_equals_rotor"]
assert checks["rotor_log_exp_round_trip"]
manifest = remember(
    save_json(
        {**checks, "artifacts": [rel(path) for path in artifact_paths]},
        TOPIC,
        "checks",
        "final-sanity-checks.json",
        root=ARTIFACT_ROOT,
    )
)
print(rel(manifest))
checks
